# Connectivity 4 — Co-regulation of coupling parameters

| | |
|---|---|
| **Original** | `Liam_work/tests/graphical_analysis/AD_Degeneracy_Coregulation.ipynb` |
| **Status** | Side project (Feb 2026). Follow-up to Connectivity 3. |

Same Marder-lab framing, but uses the richer within-animal data (hundreds of 50 ms coupling filters per
animal): Schulz-style pairwise co-regulation, temporal structure of coupling filters, Prinz ratio
constraints, cross-region coordination, and an age × genotype interaction test.

Reads `data/glm_results/glm_connectivity_results_20animals_REM.pkl`.

Paths: raw inputs are read from `/media/HlabShare/...` (read-only). Anything this copy writes goes under `/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/` — never back into `Liam_work/`.

---
*Repo copy: outputs stripped, header/section notes added, lab preamble added where missing, and output
paths moved into this repo. Analysis logic is unchanged from Liam's version.*

In [ ]:
# =============================================================================
# LAB-WIDE PREAMBLE — CPU thread limits + figure-export font config
# (added in the repo copy; set BEFORE numpy/scipy/sklearn/xgboost are imported)
# =============================================================================
import os
import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family': 'DejaVu Sans'})

# Limit CPU threads — shared server "marvin"
os.environ['OMP_NUM_THREADS']        = '4'
os.environ['OPENBLAS_NUM_THREADS']   = '4'
os.environ['MKL_NUM_THREADS']        = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS']    = '4'

# Degeneracy & Co-regulation in AD Functional Connectivity

**Framework**: Schulz et al. 2006 found that identified neurons across animals show 2-4x variability
in absolute ion channel expression, but **strong pairwise correlations** between channels (r² = 0.7-0.9).
Prinz et al. 2004 found that valid pyloric networks are scattered in absolute parameter space but
**constrained in ratio space**.

**Our data**: GLM-inferred coupling weights + 50ms temporal coupling filters from REM sleep recordings.
Each animal has hundreds of coupling filters — rich within-animal data, not just summary statistics.

**Key test**: Not "is coupling strength different in AD?" but **"is the co-regulation structure
of coupling parameters disrupted in AD?"**

**Age × Genotype**: APP/PS1 mice develop amyloid plaques ~6-7 months. Young AD = pre-plaque genotype.
Old AD = plaques + aging. Young and old WT brains differ fundamentally. All analyses are age-stratified.

| Group | n | Ages |
|-------|---|------|
| Young WT | 5 | 99-207d |
| Young AD | 4 | 73-112d |
| Old WT | 5 | 358-532d |
| Old AD | 4 | 381-492d |

In [ ]:
# =============================================================================
# SETUP: Load data, define groups, extract rich features per animal
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from math import comb
from itertools import combinations
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
np.random.seed(42)

RESULTS_FILE = '/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/data/glm_results/glm_connectivity_results_20animals_REM.pkl'  # repo copy
REGION_MAP = {'ACad': 'AC', 'ACC': 'AC', 'CA1': 'CA1', 'RSPv': 'RSC', 'RSC': 'RSC', 'V1': 'V1'}
REGIONS = ['AC', 'CA1', 'RSC', 'V1']
MIN_NEURONS = 7
AGE_CUTOFF = 212

EXCLUDE = {
    'ZBR113': '38 neurons after rescoring (>2 SD below mean)',
    'CAF104': 'spike sorting instability (81% neurons with ISI >100s)'
}

ANIMAL_AGES = {
    'BTH100': 98.7, 'BTH90': 101.0, 'BTH95': 105.4, 'CAF42': 207.4,
    'CAF69': 358.2, 'CAF71': 492.3, 'CAF73': 381.3, 'CAF74': 419.4,
    'CAF75': 388.5, 'CAF77': 394.3, 'CAF81': 455.2, 'CAF82': 455.2,
    'CAF88': 532.4, 'CAF104': 387.2, 'ZBR106': 99.0, 'ZBR113': 91.0,
    'ZBR114': 73.0, 'ZBR117': 79.0, 'ZBR118': 99.0, 'ZBR119': 112.0,
}

with open(RESULTS_FILE, 'rb') as f:
    glm_data = pickle.load(f)

print(f'Loaded {len(glm_data)} animals')
for name, reason in EXCLUDE.items():
    print(f'  EXCLUDING {name}: {reason}')

# ── Group definitions ──
group_order = ['YOUNG_WT', 'YOUNG_AD', 'OLD_WT', 'OLD_AD']
group_colors = {'YOUNG_WT': 'steelblue', 'YOUNG_AD': 'lightcoral',
                'OLD_WT': 'darkblue', 'OLD_AD': 'darkred'}
group_markers = {'YOUNG_WT': 'o', 'YOUNG_AD': 'o', 'OLD_WT': 's', 'OLD_AD': 's'}

# ── Extract RICH features per animal ──
# This is the key difference from the old notebook: we keep the full coupling
# filter data, not just 3 summary metrics per region.

animals = {}

for name, res in glm_data.items():
    if name in EXCLUDE:
        continue
    
    age = ANIMAL_AGES.get(name, np.nan)
    age_group = 'YOUNG' if age <= AGE_CUTOFF else 'OLD'
    group = f'{age_group}_{res["genotype"]}'
    
    a = {
        'name': name,
        'genotype': res['genotype'],
        'age': age,
        'age_group': age_group,
        'group': group,
        'n_neurons': res['n_neurons'],
        'neuron_info': res['neuron_info'],
    }
    
    # Per-region data: keep connectivity matrices + coupling filters
    a['regions'] = {}
    for reg_raw, rdata in res['regions'].items():
        reg = REGION_MAP.get(reg_raw, reg_raw)
        if reg not in REGIONS:
            continue
        n = rdata['connectivity'].shape[0]
        if n < MIN_NEURONS:
            continue
        
        conn = rdata['connectivity']
        mask = ~np.eye(n, dtype=bool)
        weights = conn[mask]
        nonzero_mask = np.abs(weights) > 1e-6
        nonzero = weights[nonzero_mask]
        
        # Get neuron info for this region
        idx = rdata['neuron_indices']
        region_info = [res['neuron_info'][i] for i in idx]
        cell_types = [ni['cell_type'] for ni in region_info]
        firing_rates = [ni['firing_rate_hz'] for ni in region_info]
        
        # Cell-type-specific connectivity
        n_rsu = sum(1 for ct in cell_types if ct == 'RSU')
        n_fs = sum(1 for ct in cell_types if ct == 'FS')
        
        # Build cell-type pair coupling matrices
        ct_strengths = {}
        for pair in ['RSU_RSU', 'RSU_FS', 'FS_RSU', 'FS_FS']:
            ct_strengths[pair] = []
        
        for si in range(n):
            for ti in range(n):
                if si == ti:
                    continue
                w = conn[si, ti]
                if abs(w) < 1e-6:
                    continue
                src_ct = cell_types[si]
                tgt_ct = cell_types[ti]
                pair_key = f'{src_ct}_{tgt_ct}'
                if pair_key in ct_strengths:
                    ct_strengths[pair_key].append(w)
        
        # Coupling filter analysis
        filters = rdata.get('coupling_filters', {})
        filter_params = []
        for (src, tgt), filt in filters.items():
            if np.abs(filt).max() < 1e-8:
                continue  # zero filter
            peak_idx = np.argmax(np.abs(filt))
            peak_val = filt[peak_idx]
            peak_latency = peak_idx + 1  # ms (1-indexed)
            strength = np.linalg.norm(filt)
            # Sign: excitatory if peak is positive
            sign = np.sign(peak_val)
            filter_params.append({
                'peak_latency': peak_latency,
                'peak_val': peak_val,
                'strength': strength,
                'sign': sign,
                'filter': filt,
            })
        
        a['regions'][reg] = {
            'connectivity': conn,
            'n_neurons': n,
            'n_rsu': n_rsu,
            'n_fs': n_fs,
            'nonzero_weights': nonzero,
            'density': len(nonzero) / len(weights),
            'mean_strength': np.mean(np.abs(nonzero)) if len(nonzero) > 0 else 0,
            'ei_ratio': np.sum(nonzero > 0) / max(np.sum(nonzero < 0), 1),
            'ct_strengths': ct_strengths,
            'firing_rates': firing_rates,
            'cell_types': cell_types,
            'filter_params': filter_params,
            'n_filters_nonzero': len(filter_params),
        }
    
    animals[name] = a

# ── Print animal summary ──
print(f'\n{"Animal":<10} {"Group":<12} {"Age":>5}  Regions  Filters')
print('-' * 65)
for grp in group_order:
    members = [(n, a) for n, a in sorted(animals.items()) if a['group'] == grp]
    for name, a in sorted(members, key=lambda x: x[1]['age']):
        regs = list(a['regions'].keys())
        n_filt = sum(r['n_filters_nonzero'] for r in a['regions'].values())
        print(f'{name:<10} {grp:<12} {a["age"]:>5.0f}d  {",".join(regs):<16} {n_filt} filters')
    print()

group_size_strs = []
for g in group_order:
    n = sum(1 for a in animals.values() if a['group'] == g)
    group_size_strs.append(g + '=' + str(n))
print('Group sizes: ' + ', '.join(group_size_strs))
total_filters = sum(sum(r['n_filters_nonzero'] for r in a['regions'].values()) for a in animals.values())
print(f'Total non-zero coupling filters across all animals: {total_filters}')

# ── Helper functions ──
def exact_perm_test(x, y, n_perm=50000):
    """Two-sided permutation test on difference of means + Cliff's delta."""
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan, np.nan, np.nan
    obs_diff = np.mean(x) - np.mean(y)
    pooled = np.concatenate([x, y])
    count = 0
    for _ in range(n_perm):
        perm = np.random.permutation(pooled)
        d = np.mean(perm[:nx]) - np.mean(perm[nx:])
        if abs(d) >= abs(obs_diff):
            count += 1
    p = (count + 1) / (n_perm + 1)
    greater = sum(1 for xi in x for yi in y if xi > yi)
    lesser = sum(1 for xi in x for yi in y if xi < yi)
    cliff_d = (greater - lesser) / (nx * ny)
    return obs_diff, p, cliff_d

In [ ]:
# =============================================================================
# ANALYSIS 1: SCHULZ CO-REGULATION
#
# Schulz 2006/2007: Measured 6 ion channel mRNAs in identified neurons across
# ~20 crabs. Found strong PAIRWISE CORRELATIONS between channels:
#   LP neurons: shal vs shaw r=0.965, PD neurons: IH vs para r=0.913
# The x-axis is one channel, y-axis is another, each point is one animal.
# The CORRELATION is the co-regulation. Channels scale together.
#
# Our analog: Within each animal, we have hundreds of coupling filters.
# Each filter has properties: strength, peak latency, sign.
# Schulz asked: do channels co-regulate within a neuron type?
# We ask: do coupling properties co-regulate within an animal's network?
#
# Method: For each animal, compute within-animal Pearson correlations between
# filter properties (e.g., r(strength, peak_latency) across all filters).
# This within-animal correlation is a scalar per animal.
# Then compare these correlations across groups.
#
# If WT brains co-regulate their coupling dynamics (strong filters have
# consistent timing), the within-animal correlations should be structured.
# AD may disrupt this co-regulation.
# =============================================================================
print('=' * 70)
print('ANALYSIS 1: Schulz Co-Regulation — Within-Animal Coupling Correlations')
print('=' * 70)
print('Schulz: shal vs shaw r=0.965 in LP neurons across 20 crabs.')
print('Us: coupling strength vs peak latency across hundreds of filters per animal.')

# ── 1A: Within-animal correlations between filter properties ──
# For each animal, pool all non-zero filters and compute pairwise correlations
# between filter properties. These are the "channel-channel" correlations.

schulz_pairs = [
    ('strength', 'peak_latency', 'Strength vs Peak Latency',
     'Do stronger connections have faster/slower temporal dynamics?'),
    ('strength', 'firing_rate', 'Strength vs Source Firing Rate',
     'Do neurons that fire more have stronger outgoing coupling?'),
    ('peak_latency', 'firing_rate', 'Peak Latency vs Source Firing Rate',
     'Do fast-firing neurons have different synaptic timing?'),
]

within_animal_corrs = {}  # animal → {pair_name: r}

for name, a in animals.items():
    corrs = {}
    
    # Pool all non-zero filters across regions with source neuron info
    all_strengths = []
    all_latencies = []
    all_source_fr = []
    
    for reg, rd in a['regions'].items():
        idx_list = list(range(rd['n_neurons']))
        for fp in rd['filter_params']:
            all_strengths.append(fp['strength'])
            all_latencies.append(fp['peak_latency'])
        # Source firing rates — match to filters via region
        for fr in rd['firing_rates']:
            all_source_fr.append(fr)
    
    # For strength vs latency: use filter-level data (hundreds of points)
    s = np.array(all_strengths)
    l = np.array(all_latencies)
    
    if len(s) >= 20:
        corrs['strength_vs_latency'] = stats.pearsonr(s, l)[0]
        corrs['n_filters'] = len(s)
    
    # For region-level correlations: use per-region summaries
    # (this is closer to Schulz — different "channels" measured in same "neuron")
    reg_densities = []
    reg_strengths = []
    reg_ei_ratios = []
    reg_cvs = []
    reg_lat_medians = []
    reg_frs = []
    
    for reg in REGIONS:
        if reg not in a['regions']:
            continue
        rd = a['regions'][reg]
        reg_densities.append(rd['density'])
        reg_strengths.append(rd['mean_strength'])
        reg_ei_ratios.append(rd['ei_ratio'])
        w = np.abs(rd['nonzero_weights'])
        reg_cvs.append(np.std(w) / np.mean(w) if len(w) > 5 else np.nan)
        lats = [fp['peak_latency'] for fp in rd['filter_params']]
        reg_lat_medians.append(np.median(lats) if len(lats) > 5 else np.nan)
        reg_frs.append(np.mean(rd['firing_rates']))
    
    # Cross-parameter correlations across regions (Schulz-style)
    # Each region is like a "cell" and each metric is like a "channel"
    if len(reg_densities) >= 3:
        r_ds, _ = stats.pearsonr(reg_densities, reg_strengths)
        corrs['density_vs_strength'] = r_ds
        
        r_de, _ = stats.pearsonr(reg_densities, reg_ei_ratios)
        corrs['density_vs_ei'] = r_de
        
        r_se, _ = stats.pearsonr(reg_strengths, reg_ei_ratios)
        corrs['strength_vs_ei'] = r_se
        
        valid = ~np.isnan(reg_lat_medians)
        if sum(valid) >= 3:
            r_sl, _ = stats.pearsonr(np.array(reg_strengths)[valid],
                                      np.array(reg_lat_medians)[valid])
            corrs['strength_vs_median_lat'] = r_sl
        
        r_sf, _ = stats.pearsonr(reg_strengths, reg_frs)
        corrs['strength_vs_fr'] = r_sf
    
    within_animal_corrs[name] = corrs

# ── Print and test each correlation pair ──
corr_pairs_to_test = [
    ('strength_vs_latency', 'Filter strength vs peak latency'),
    ('density_vs_strength', 'Region density vs strength (across regions)'),
    ('density_vs_ei', 'Region density vs E/I ratio'),
    ('strength_vs_ei', 'Region strength vs E/I ratio'),
    ('strength_vs_median_lat', 'Region strength vs median filter latency'),
    ('strength_vs_fr', 'Region strength vs firing rate'),
]

print(f'\n{"Correlation pair":<45} {"Age":>5} {"WT mean r":>10} {"AD mean r":>10} {"p":>8} {"d":>6}')
print('-' * 90)

schulz_results = []
for corr_key, corr_label in corr_pairs_to_test:
    for age in ['YOUNG', 'OLD']:
        wt_vals = [within_animal_corrs[n][corr_key] for n in animals
                   if animals[n]['group'] == f'{age}_WT' and corr_key in within_animal_corrs[n]]
        ad_vals = [within_animal_corrs[n][corr_key] for n in animals
                   if animals[n]['group'] == f'{age}_AD' and corr_key in within_animal_corrs[n]]
        
        if len(wt_vals) >= 2 and len(ad_vals) >= 2:
            diff, p, cliff_d = exact_perm_test(np.array(wt_vals), np.array(ad_vals))
            sig = '*' if p < 0.05 else ''
            print(f'{corr_label:<45} {age:>5} {np.mean(wt_vals):>+10.3f} {np.mean(ad_vals):>+10.3f} '
                  f'{p:>8.4f} {cliff_d:>+6.2f} {sig}')
            schulz_results.append({
                'pair': corr_key, 'label': corr_label, 'age': age,
                'wt_mean': np.mean(wt_vals), 'ad_mean': np.mean(ad_vals),
                'p': p, 'cliff_d': cliff_d
            })

# ── Visualize the within-animal correlations ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (corr_key, corr_label) in enumerate(corr_pairs_to_test):
    ax = axes[idx]
    
    data_by_group = []
    for grp in group_order:
        vals = [within_animal_corrs[n][corr_key] for n in animals
                if animals[n]['group'] == grp and corr_key in within_animal_corrs[n]]
        data_by_group.append(vals)
    
    positions = [0, 1, 3, 4]
    bp = ax.boxplot(data_by_group, positions=positions, widths=0.5,
                    patch_artist=True, showfliers=False)
    for i, grp in enumerate(group_order):
        bp['boxes'][i].set_facecolor(group_colors[grp])
        bp['boxes'][i].set_alpha(0.35)
        vals = data_by_group[i]
        if vals:
            jitter = np.random.normal(0, 0.06, len(vals))
            ax.scatter(positions[i] + jitter, vals, c=group_colors[grp],
                       s=50, edgecolor='k', linewidth=0.4, zorder=3)
    
    ax.set_xticks(positions)
    ax.set_xticklabels(['Y-WT', 'Y-AD', 'O-WT', 'O-AD'], fontsize=9)
    ax.set_ylabel('Within-animal r')
    ax.set_title(corr_label, fontweight='bold', fontsize=9)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')

plt.suptitle('Schulz Test: Within-Animal Coupling Parameter Correlations\n'
             'Each point = one animal\'s correlation between two coupling properties',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── 1B: Schulz-style scatter plots for the strongest correlations ──
# Show actual filter-level data for a few example animals
print('\n--- Example Schulz Scatter Plots (filter strength vs peak latency) ---')

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

for row, age in enumerate(['YOUNG', 'OLD']):
    # Get one WT and one AD example from each age group
    wt_examples = [n for n in animals if animals[n]['group'] == f'{age}_WT'][:2]
    ad_examples = [n for n in animals if animals[n]['group'] == f'{age}_AD'][:2]
    examples = wt_examples + ad_examples
    
    for col, name in enumerate(examples[:4]):
        ax = axes[row, col]
        a = animals[name]
        
        all_s, all_l = [], []
        for reg, rd in a['regions'].items():
            for fp in rd['filter_params']:
                all_s.append(fp['strength'])
                all_l.append(fp['peak_latency'])
        
        all_s = np.array(all_s)
        all_l = np.array(all_l)
        
        r_val = within_animal_corrs[name].get('strength_vs_latency', np.nan)
        
        ax.scatter(all_s, all_l, s=8, alpha=0.3, c=group_colors[a['group']])
        ax.set_xlabel('Filter strength (L2 norm)')
        ax.set_ylabel('Peak latency (ms)')
        ax.set_title(f'{name} ({a["group"]})\nr = {r_val:.3f}, n = {len(all_s)} filters',
                     fontsize=9, fontweight='bold', color=group_colors[a['group']])

plt.suptitle('Schulz-Style Scatter: Each dot = one coupling filter within one animal\n'
             '(Like plotting shal vs BKKCa expression across ion channels within one neuron)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── 1C: Feature profile heatmaps (still useful for visualization) ──
print('\n--- Feature Profiles (z-scored) ---')

# Build feature matrix for profile visualization
feature_defs = []
for reg in REGIONS:
    feature_defs.append(f'{reg}_density')
    feature_defs.append(f'{reg}_mean_strength')
    feature_defs.append(f'{reg}_ei_ratio')
    feature_defs.append(f'{reg}_weight_cv')
    feature_defs.append(f'{reg}_median_peak_lat')
    feature_defs.append(f'{reg}_mean_fr')

profiles = {}
for name, a in animals.items():
    vec = {}
    for reg in REGIONS:
        if reg not in a['regions']:
            continue
        rd = a['regions'][reg]
        vec[f'{reg}_density'] = rd['density']
        vec[f'{reg}_mean_strength'] = rd['mean_strength']
        vec[f'{reg}_ei_ratio'] = rd['ei_ratio']
        w = np.abs(rd['nonzero_weights'])
        vec[f'{reg}_weight_cv'] = np.std(w) / np.mean(w) if len(w) > 5 else np.nan
        lats = [fp['peak_latency'] for fp in rd['filter_params']]
        vec[f'{reg}_median_peak_lat'] = np.median(lats) if len(lats) > 5 else np.nan
        vec[f'{reg}_mean_fr'] = np.mean(rd['firing_rates'])
    profiles[name] = vec

animal_names = sorted(animals.keys())
X_raw = np.full((len(animal_names), len(feature_defs)), np.nan)
for i, name in enumerate(animal_names):
    for j, feat in enumerate(feature_defs):
        X_raw[i, j] = profiles[name].get(feat, np.nan)

good_feats = [j for j in range(len(feature_defs)) if np.isnan(X_raw[:, j]).mean() < 0.25]
feat_names_used = [feature_defs[j] for j in good_feats]
X = X_raw[:, good_feats]
groups = np.array([animals[n]['group'] for n in animal_names])

# Impute NaN with group median
for j in range(X.shape[1]):
    for grp in group_order:
        mask = groups == grp
        col = X[mask, j]
        nan_mask = np.isnan(col)
        if nan_mask.any() and (~nan_mask).any():
            col[nan_mask] = np.nanmedian(col)
            X[mask, j] = col

X_std = (X - np.nanmean(X, axis=0)) / np.nanstd(X, axis=0)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for gi, grp in enumerate(group_order):
    ax = axes[gi // 2, gi % 2]
    grp_idx = [i for i, g in enumerate(groups) if g == grp]
    grp_names = [animal_names[i] for i in grp_idx]
    data = X_std[grp_idx]
    im = ax.imshow(data, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
    ax.set_yticks(range(len(grp_names)))
    ax.set_yticklabels(grp_names, fontsize=8)
    ax.set_xticks(range(len(feat_names_used)))
    ax.set_xticklabels([f.replace('_', '\n') for f in feat_names_used], fontsize=6, rotation=45, ha='right')
    ax.set_title(f'{grp} (n={len(grp_idx)})', fontweight='bold', color=group_colors[grp])
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Feature Profiles (z-scored): Each row = one animal',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# ANALYSIS 2: COUPLING FILTER TEMPORAL STRUCTURE
#
# This is data Marder never had. Each connection has a 50ms temporal kernel
# showing HOW neuron i affects neuron j over time. These filters encode
# synaptic dynamics — monosynaptic excitation peaks at 2-5ms, feedforward
# inhibition at 3-8ms, etc.
#
# If the brain co-regulates its circuit dynamics (Schulz framework applied
# to temporal structure), filter shapes should be consistent within WT animals.
# AD may disrupt this temporal co-regulation → more variable filter shapes.
#
# Key advantage: each animal has HUNDREDS of filters, so within-animal
# distributions are well-estimated even at n=4-5 animals per group.
# =============================================================================
print('=' * 70)
print('ANALYSIS 2: Coupling Filter Temporal Structure')
print('=' * 70)

# ── 2A: Within-animal filter consistency ──
# For each animal, compute the CV of peak latencies across all non-zero filters.
# Also compute the mean filter shape and its variability.

print('\n--- 2A: Within-Animal Filter Consistency ---')
print('Lower CV of peak latency = more stereotyped synaptic dynamics')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

filter_consistency = {}  # animal → {region → stats}

for name, a in animals.items():
    filter_consistency[name] = {}
    all_lats = []
    all_strengths = []
    
    for reg, rd in a['regions'].items():
        fps = rd['filter_params']
        if len(fps) < 10:
            continue
        
        lats = np.array([fp['peak_latency'] for fp in fps])
        strengths = np.array([fp['strength'] for fp in fps])
        
        filter_consistency[name][reg] = {
            'lat_cv': np.std(lats) / np.mean(lats),
            'lat_mean': np.mean(lats),
            'lat_median': np.median(lats),
            'strength_cv': np.std(strengths) / np.mean(strengths),
            'n_filters': len(fps),
        }
        all_lats.extend(lats)
        all_strengths.extend(strengths)
    
    if len(all_lats) > 10:
        filter_consistency[name]['overall'] = {
            'lat_cv': np.std(all_lats) / np.mean(all_lats),
            'strength_cv': np.std(all_strengths) / np.mean(all_strengths),
            'n_filters': len(all_lats),
            'lat_mean': np.mean(all_lats),
        }

# Compare latency CV by group
cv_results = []
for metric_key, metric_label in [('lat_cv', 'Peak Latency CV'), ('strength_cv', 'Strength CV')]:
    ax = axes[0] if 'lat' in metric_key else axes[1]
    
    print(f'\n{metric_label}:')
    
    for age in ['YOUNG', 'OLD']:
        wt_vals = [filter_consistency[n]['overall'][metric_key]
                   for n in animals if animals[n]['group'] == f'{age}_WT'
                   and 'overall' in filter_consistency[n]]
        ad_vals = [filter_consistency[n]['overall'][metric_key]
                   for n in animals if animals[n]['group'] == f'{age}_AD'
                   and 'overall' in filter_consistency[n]]
        
        if len(wt_vals) >= 2 and len(ad_vals) >= 2:
            diff, p, cliff_d = exact_perm_test(np.array(wt_vals), np.array(ad_vals))
            sig = '*' if p < 0.05 else ''
            print(f'  {age}: WT={np.mean(wt_vals):.3f}+/-{np.std(wt_vals):.3f} (n={len(wt_vals)}), '
                  f'AD={np.mean(ad_vals):.3f}+/-{np.std(ad_vals):.3f} (n={len(ad_vals)}), '
                  f'p={p:.4f}, d={cliff_d:+.2f} {sig}')
            cv_results.append({'metric': metric_key, 'age': age, 'p': p, 'cliff_d': cliff_d,
                               'wt_mean': np.mean(wt_vals), 'ad_mean': np.mean(ad_vals)})
    
    # Box plot
    data_by_group = []
    for grp in group_order:
        vals = [filter_consistency[n]['overall'][metric_key]
                for n in animals if animals[n]['group'] == grp
                and 'overall' in filter_consistency[n]]
        data_by_group.append(vals)
    
    positions = [0, 1, 3, 4]
    bp = ax.boxplot(data_by_group, positions=positions, widths=0.5,
                    patch_artist=True, showfliers=False)
    for i, grp in enumerate(group_order):
        bp['boxes'][i].set_facecolor(group_colors[grp])
        bp['boxes'][i].set_alpha(0.35)
    for i, grp in enumerate(group_order):
        vals = data_by_group[i]
        if vals:
            jitter = np.random.normal(0, 0.06, len(vals))
            ax.scatter(positions[i] + jitter, vals, c=group_colors[grp],
                       s=50, edgecolor='k', linewidth=0.4, zorder=3)
    
    ax.set_xticks(positions)
    ax.set_xticklabels(['Y-WT', 'Y-AD', 'O-WT', 'O-AD'], fontsize=9)
    ax.set_ylabel(metric_label)
    ax.set_title(metric_label, fontweight='bold')

plt.suptitle('Filter Consistency: Higher CV = more variable synaptic dynamics',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── 2B: Average coupling filter shapes by group ──
# Overlay the mean filter shape for each group. If WT is co-regulated,
# the mean filter should be similar across WT animals. AD may show
# different or noisier shapes.

print('\n--- 2B: Mean Coupling Filter Shapes by Group ---')

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for ri, reg in enumerate(REGIONS):
    # Top row: excitatory filters (sign > 0)
    # Bottom row: inhibitory filters (sign < 0)
    for row, (sign_label, sign_val) in enumerate([('Excitatory', 1), ('Inhibitory', -1)]):
        ax = axes[row, ri]
        
        for grp in group_order:
            grp_filters = []
            for name, a in animals.items():
                if a['group'] != grp or reg not in a['regions']:
                    continue
                for fp in a['regions'][reg]['filter_params']:
                    if fp['sign'] == sign_val:
                        grp_filters.append(fp['filter'])
            
            if len(grp_filters) < 5:
                continue
            
            grp_filters = np.array(grp_filters)
            mean_filt = np.mean(grp_filters, axis=0)
            sem_filt = np.std(grp_filters, axis=0) / np.sqrt(len(grp_filters))
            
            t = np.arange(1, 51)
            ax.plot(t, mean_filt, color=group_colors[grp], linewidth=2, label=f'{grp} (n={len(grp_filters)})')
            ax.fill_between(t, mean_filt - sem_filt, mean_filt + sem_filt,
                            color=group_colors[grp], alpha=0.15)
        
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xlabel('Lag (ms)' if row == 1 else '')
        if ri == 0:
            ax.set_ylabel(f'{sign_label}\nCoupling weight')
        ax.set_title(f'{reg} — {sign_label}' if row == 0 else '', fontweight='bold')
        if ri == 3 and row == 0:
            ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Mean Coupling Filter Shapes by Group (± SEM across individual filters)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── 2C: Filter peak latency distributions ──
# The temporal structure of connections. Monosynaptic = fast peak (1-5ms).
# Polysynaptic = slower peak (>10ms). Is the distribution different in AD?

print('\n--- 2C: Filter Peak Latency Distributions ---')

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ri, reg in enumerate(REGIONS):
    ax = axes[ri]
    
    for grp in group_order:
        all_lats = []
        for name, a in animals.items():
            if a['group'] != grp or reg not in a['regions']:
                continue
            all_lats.extend([fp['peak_latency'] for fp in a['regions'][reg]['filter_params']])
        
        if len(all_lats) > 10:
            ax.hist(all_lats, bins=np.arange(0.5, 51.5, 2), density=True,
                    alpha=0.3, color=group_colors[grp], label=f'{grp} (n={len(all_lats)})')
            # KDE
            from scipy.stats import gaussian_kde
            kde = gaussian_kde(all_lats, bw_method=0.3)
            x = np.linspace(1, 50, 200)
            ax.plot(x, kde(x), color=group_colors[grp], linewidth=2)
    
    ax.set_xlabel('Peak latency (ms)')
    ax.set_ylabel('Density' if ri == 0 else '')
    ax.set_title(reg, fontweight='bold')
    if ri == 3:
        ax.legend(fontsize=7)

plt.suptitle('Distribution of Coupling Filter Peak Latencies',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# ANALYSIS 3: PRINZ RATIO CONSTRAINTS — Cell-type coupling ratios
#
# Prinz showed that valid pyloric networks are scattered in absolute
# conductance space but CONSTRAINED in ratio space. The ratio between
# synaptic conductances matters more than their absolute values.
#
# Our analog: within each animal, compute ratios of cell-type-specific
# coupling strengths (e.g., RSU→RSU / FS→RSU). If the brain maintains
# E/I balance through co-regulation, these ratios should be constrained
# in WT but potentially unconstrained (higher variance) in AD.
#
# Each animal has tens to hundreds of connections per cell-type pair,
# so ratio estimates are robust within each animal.
# =============================================================================
print('=' * 70)
print('ANALYSIS 3: Prinz Ratio Constraints — Cell-Type Coupling Ratios')
print('=' * 70)

# For each animal, compute mean |coupling| by cell-type pair per region
# Then take ratios: RSU→RSU / FS→RSU (excitation/inhibition balance proxy)

# First, build per-animal cell-type coupling summary
ct_summary = {}
for name, a in animals.items():
    ct_summary[name] = {}
    for reg, rd in a['regions'].items():
        ct_s = {}
        for pair_key, weights in rd['ct_strengths'].items():
            if len(weights) >= 3:
                ct_s[pair_key] = {
                    'mean_abs': np.mean(np.abs(weights)),
                    'n': len(weights),
                    'mean_signed': np.mean(weights),
                }
        ct_summary[name][reg] = ct_s

# Define biologically meaningful ratios
# These are like Prinz's conductance ratios — excitation vs inhibition balance
ratio_defs = [
    ('RSU_RSU', 'FS_RSU', 'RSU→RSU / FS→RSU', 'Excitation-to-Inhibition balance'),
    ('RSU_RSU', 'RSU_FS', 'RSU→RSU / RSU→FS', 'Recurrent exc. vs feedforward to FS'),
    ('FS_RSU', 'FS_FS', 'FS→RSU / FS→FS', 'Inhibition target selectivity'),
]

print('\n--- Ratio Values and Variance by Age x Genotype ---')
print('Prinz prediction: WT ratios should be constrained (low variance).')
print('If AD loses co-regulation, ratio variance increases.')

fig, axes = plt.subplots(len(ratio_defs), 2, figsize=(14, 4 * len(ratio_defs)))

ratio_results = []

for ri, (num_pair, den_pair, ratio_label, bio_label) in enumerate(ratio_defs):
    # Compute log-ratio per animal (average across regions that have both)
    animal_ratios = {}
    for name, a in animals.items():
        log_ratios = []
        for reg in a['regions']:
            ct_s = ct_summary[name].get(reg, {})
            if num_pair in ct_s and den_pair in ct_s:
                num = ct_s[num_pair]['mean_abs']
                den = ct_s[den_pair]['mean_abs']
                if den > 1e-8:
                    log_ratios.append(np.log(num / den))
        if log_ratios:
            animal_ratios[name] = np.mean(log_ratios)
    
    # Split by group
    group_vals = {g: [] for g in group_order}
    for name, val in animal_ratios.items():
        group_vals[animals[name]['group']].append(val)
    
    # Left: ratio values
    ax = axes[ri, 0]
    positions = [0, 1, 3, 4]
    data = [group_vals[g] for g in group_order]
    
    bp = ax.boxplot(data, positions=positions, widths=0.5,
                    patch_artist=True, showfliers=False)
    for i, grp in enumerate(group_order):
        bp['boxes'][i].set_facecolor(group_colors[grp])
        bp['boxes'][i].set_alpha(0.35)
        vals = data[i]
        if vals:
            jitter = np.random.normal(0, 0.06, len(vals))
            ax.scatter(positions[i] + jitter, vals, c=group_colors[grp],
                       s=50, edgecolor='k', linewidth=0.4, zorder=3)
    
    ax.set_xticks(positions)
    ax.set_xticklabels(['Y-WT', 'Y-AD', 'O-WT', 'O-AD'])
    ax.set_ylabel('log(ratio)')
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title(f'{ratio_label}\n({bio_label})', fontweight='bold', fontsize=10)
    
    # Right: variance comparison (bar chart)
    ax2 = axes[ri, 1]
    
    print(f'\n{ratio_label} ({bio_label}):')
    
    for age in ['YOUNG', 'OLD']:
        wt_v = np.array(group_vals[f'{age}_WT'])
        ad_v = np.array(group_vals[f'{age}_AD'])
        
        if len(wt_v) >= 2 and len(ad_v) >= 2:
            # Test ratio values
            diff, p_val, cliff_d = exact_perm_test(wt_v, ad_v)
            
            # Test ratio variance (Levene's)
            var_wt = np.var(wt_v, ddof=1)
            var_ad = np.var(ad_v, ddof=1)
            var_ratio = var_ad / var_wt if var_wt > 1e-10 else np.nan
            
            if len(wt_v) >= 3 and len(ad_v) >= 3:
                _, p_var = stats.levene(wt_v, ad_v, center='median')
            else:
                p_var = np.nan
            
            sig_v = '*' if p_val < 0.05 else ''
            sig_var = '*' if p_var < 0.05 else ''
            
            print(f'  {age}: WT={np.mean(wt_v):.3f}+/-{np.std(wt_v):.3f}, '
                  f'AD={np.mean(ad_v):.3f}+/-{np.std(ad_v):.3f}, '
                  f'p_val={p_val:.3f}{sig_v}, '
                  f'var_ratio={var_ratio:.2f}x, p_var={p_var:.3f}{sig_var}')
            
            ratio_results.append({
                'ratio': ratio_label, 'age': age,
                'p_val': p_val, 'cliff_d': cliff_d,
                'var_wt': var_wt, 'var_ad': var_ad, 'var_ratio': var_ratio,
                'p_var': p_var,
            })
    
    # Variance bar chart
    bar_data = []
    bar_labels = []
    bar_colors_list = []
    for age in ['YOUNG', 'OLD']:
        for geno in ['WT', 'AD']:
            grp = f'{age}_{geno}'
            v = group_vals[grp]
            bar_data.append(np.var(v, ddof=1) if len(v) > 1 else 0)
            bar_labels.append(f'{age[:1]}-{geno}')
            bar_colors_list.append(group_colors[grp])
    
    ax2.bar(range(len(bar_data)), bar_data, color=bar_colors_list, alpha=0.6)
    ax2.set_xticks(range(len(bar_labels)))
    ax2.set_xticklabels(bar_labels)
    ax2.set_ylabel('Variance of log(ratio)')
    ax2.set_title('Ratio Variance\n(Prinz: AD should be higher)', fontweight='bold', fontsize=10)

plt.suptitle('Cell-Type Coupling Ratios: Values (left) and Variance (right)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Prinz-style scatter: plot animals in ratio space ──
print('\n--- Prinz Scatter: Animals in Ratio Space ---')

# Build ratio vectors per animal (one dimension per ratio)
ratio_matrix = []
ratio_names_used = []
ratio_animal_names = []

for name in sorted(animals.keys()):
    a = animals[name]
    ratios = []
    valid = True
    for num_pair, den_pair, _, _ in ratio_defs:
        log_rats = []
        for reg in a['regions']:
            ct_s = ct_summary[name].get(reg, {})
            if num_pair in ct_s and den_pair in ct_s:
                num = ct_s[num_pair]['mean_abs']
                den = ct_s[den_pair]['mean_abs']
                if den > 1e-8:
                    log_rats.append(np.log(num / den))
        if log_rats:
            ratios.append(np.mean(log_rats))
        else:
            valid = False
            break
    if valid:
        ratio_matrix.append(ratios)
        ratio_animal_names.append(name)

ratio_matrix = np.array(ratio_matrix)
ratio_groups = np.array([animals[n]['group'] for n in ratio_animal_names])

if ratio_matrix.shape[0] >= 6 and ratio_matrix.shape[1] >= 2:
    # PCA on ratio space
    ratio_std = (ratio_matrix - ratio_matrix.mean(axis=0)) / ratio_matrix.std(axis=0)
    pca_r = PCA()
    ratio_pc = pca_r.fit_transform(ratio_std)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    for grp in group_order:
        mask = ratio_groups == grp
        ax.scatter(ratio_pc[mask, 0], ratio_pc[mask, 1],
                   c=group_colors[grp], s=100, edgecolor='k', linewidth=0.5,
                   marker=group_markers[grp], label=grp, zorder=3)
    
    for i, name in enumerate(ratio_animal_names):
        ax.annotate(name, (ratio_pc[i, 0], ratio_pc[i, 1]), fontsize=7, alpha=0.7,
                    xytext=(4, 4), textcoords='offset points')
    
    # 95% confidence ellipses
    for grp in group_order:
        mask = ratio_groups == grp
        pts = ratio_pc[mask, :2]
        if len(pts) >= 3:
            cov = np.cov(pts.T)
            eigvals, eigvecs = np.linalg.eigh(cov)
            angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
            width = 2 * np.sqrt(5.991 * max(eigvals[1], 0.01))
            height = 2 * np.sqrt(5.991 * max(eigvals[0], 0.01))
            ell = Ellipse(xy=pts.mean(axis=0), width=width, height=height,
                          angle=angle, facecolor=group_colors[grp], alpha=0.1,
                          edgecolor=group_colors[grp], linewidth=2)
            ax.add_patch(ell)
    
    ax.set_xlabel(f'PC1 ({pca_r.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca_r.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title('Animals in Cell-Type Coupling Ratio Space (Prinz Framework)',
                 fontweight='bold', fontsize=13)
    ax.legend(fontsize=10)
    ax.axhline(0, color='lightgray', linewidth=0.5)
    ax.axvline(0, color='lightgray', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    
    # PERMDISP: test if AD is more dispersed in ratio space
    for age in ['YOUNG', 'OLD']:
        wt_mask = ratio_groups == f'{age}_WT'
        ad_mask = ratio_groups == f'{age}_AD'
        
        if np.sum(wt_mask) >= 3 and np.sum(ad_mask) >= 3:
            wt_pts = ratio_pc[wt_mask, :2]
            ad_pts = ratio_pc[ad_mask, :2]
            
            wt_centroid = wt_pts.mean(axis=0)
            ad_centroid = ad_pts.mean(axis=0)
            
            wt_disp = np.mean([np.linalg.norm(p - wt_centroid) for p in wt_pts])
            ad_disp = np.mean([np.linalg.norm(p - ad_centroid) for p in ad_pts])
            
            obs_diff = ad_disp - wt_disp
            
            all_pts = np.vstack([wt_pts, ad_pts])
            n_wt = len(wt_pts)
            n_perm = 10000
            null_diffs = np.zeros(n_perm)
            for pi in range(n_perm):
                perm = np.random.permutation(len(all_pts))
                p_wt = all_pts[perm[:n_wt]]
                p_ad = all_pts[perm[n_wt:]]
                null_diffs[pi] = (np.mean(np.linalg.norm(p_ad - p_ad.mean(0), axis=1)) -
                                  np.mean(np.linalg.norm(p_wt - p_wt.mean(0), axis=1)))
            
            p_disp = (np.sum(null_diffs >= obs_diff) + 1) / (n_perm + 1)
            print(f'{age} PERMDISP: WT disp={wt_disp:.3f}, AD disp={ad_disp:.3f}, '
                  f'diff={obs_diff:+.3f}, p={p_disp:.4f}')

In [ ]:
# =============================================================================
# ANALYSIS 4: CROSS-REGION COORDINATION
#
# Schulz found that electrically coupled PD neurons (same animal) had
# coordinated channel expression. The analog here: do different brain
# regions within the same animal show coordinated connectivity patterns?
#
# If the brain maintains system-wide homeostasis, an animal with strong
# AC coupling should also have strong V1 coupling (cross-region correlation).
# AD may disrupt this coordination — each region "doing its own thing."
#
# We compute within-animal cross-region similarity (correlation of regional
# connectivity profiles), then test whether WT animals are more coordinated
# than AD animals.
# =============================================================================
print('=' * 70)
print('ANALYSIS 4: Cross-Region Coordination')
print('=' * 70)
print('Schulz analog: are different regions within the same animal coordinated?')

# For each animal with >=3 regions, compute pairwise cross-region correlation
# using the coupling weight distribution statistics

cross_reg_metrics = ['density', 'mean_strength', 'ei_ratio']

cross_reg_results = {}
for name, a in animals.items():
    regs = sorted(a['regions'].keys())
    if len(regs) < 3:
        continue
    
    # Build per-region feature vector
    reg_vectors = {}
    for reg in regs:
        rd = a['regions'][reg]
        vec = [rd['density'], rd['mean_strength'], rd['ei_ratio']]
        # Add cell-type specific info
        for ct_pair in ['RSU_RSU', 'FS_RSU', 'RSU_FS']:
            w = rd['ct_strengths'].get(ct_pair, [])
            vec.append(np.mean(np.abs(w)) if len(w) > 0 else 0)
        # Add filter stats
        fps = rd['filter_params']
        if len(fps) > 5:
            vec.append(np.median([fp['peak_latency'] for fp in fps]))
            vec.append(np.mean([fp['strength'] for fp in fps]))
        else:
            vec.extend([np.nan, np.nan])
        reg_vectors[reg] = np.array(vec)
    
    # Pairwise cross-region correlations
    reg_pairs = list(combinations(regs, 2))
    corrs = []
    for r1, r2 in reg_pairs:
        v1, v2 = reg_vectors[r1], reg_vectors[r2]
        # Remove NaN dimensions
        valid = ~(np.isnan(v1) | np.isnan(v2))
        if valid.sum() >= 3:
            r = np.corrcoef(v1[valid], v2[valid])[0, 1]
            corrs.append(r)
    
    if corrs:
        cross_reg_results[name] = {
            'mean_corr': np.mean(corrs),
            'corrs': corrs,
            'n_pairs': len(corrs),
            'n_regions': len(regs),
        }

print(f'\nAnimals with cross-region data: {len(cross_reg_results)}')

# Plot and test by age group
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

coord_test_results = []
for ax_idx, age in enumerate(['YOUNG', 'OLD']):
    ax = axes[ax_idx]
    
    wt_vals = [cross_reg_results[n]['mean_corr'] for n in animals
               if n in cross_reg_results and animals[n]['group'] == f'{age}_WT']
    ad_vals = [cross_reg_results[n]['mean_corr'] for n in animals
               if n in cross_reg_results and animals[n]['group'] == f'{age}_AD']
    
    wt_names = [n for n in animals
               if n in cross_reg_results and animals[n]['group'] == f'{age}_WT']
    ad_names = [n for n in animals
               if n in cross_reg_results and animals[n]['group'] == f'{age}_AD']
    
    print(f'\n{age}:')
    for n, v in zip(wt_names, wt_vals):
        print(f'  {n} (WT): mean cross-region r = {v:.3f}')
    for n, v in zip(ad_names, ad_vals):
        print(f'  {n} (AD): mean cross-region r = {v:.3f}')
    
    if len(wt_vals) >= 2 and len(ad_vals) >= 2:
        diff, p, cliff_d = exact_perm_test(np.array(wt_vals), np.array(ad_vals))
        print(f'  WT mean={np.mean(wt_vals):.3f}, AD mean={np.mean(ad_vals):.3f}, '
              f'p={p:.4f}, d={cliff_d:+.2f}')
        coord_test_results.append({'age': age, 'p': p, 'cliff_d': cliff_d})
    
    # Plot individual animals
    all_vals = []
    all_colors = []
    all_labels = []
    
    for n in wt_names:
        all_vals.append(cross_reg_results[n]['mean_corr'])
        all_colors.append(group_colors[f'{age}_WT'])
        all_labels.append(n)
    for n in ad_names:
        all_vals.append(cross_reg_results[n]['mean_corr'])
        all_colors.append(group_colors[f'{age}_AD'])
        all_labels.append(n)
    
    x_pos = list(range(len(all_vals)))
    bars = ax.bar(x_pos, all_vals, color=all_colors, alpha=0.7, edgecolor='k', linewidth=0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(all_labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Mean cross-region correlation')
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title(f'{age}: Cross-Region Coordination\n(higher = more coordinated brain)',
                 fontweight='bold')
    
    # Add WT/AD labels
    n_wt = len(wt_names)
    if n_wt > 0:
        ax.axvspan(-0.5, n_wt - 0.5, alpha=0.05, color='blue')
    if len(ad_names) > 0:
        ax.axvspan(n_wt - 0.5, n_wt + len(ad_names) - 0.5, alpha=0.05, color='red')

plt.suptitle('Cross-Region Coordination: Do brain regions covary within each animal?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Detailed cross-region correlation matrices per group ──
print('\n--- Cross-Region Correlation Matrices by Group ---')

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for gi, grp in enumerate(group_order):
    ax = axes[gi]
    
    grp_animals = [n for n in animals if animals[n]['group'] == grp and n in cross_reg_results]
    if not grp_animals:
        ax.set_title(f'{grp}: insufficient data')
        continue
    
    # Average the region feature vectors across animals in this group
    # to get a group-level cross-region correlation matrix
    all_reg_vecs = {reg: [] for reg in REGIONS}
    for name in grp_animals:
        a = animals[name]
        for reg in REGIONS:
            if reg in a['regions']:
                rd = a['regions'][reg]
                vec = [rd['density'], rd['mean_strength'], rd['ei_ratio']]
                all_reg_vecs[reg].append(vec)
    
    # Build correlation matrix between regions
    valid_regs = [r for r in REGIONS if len(all_reg_vecs[r]) >= 2]
    n_reg = len(valid_regs)
    
    if n_reg >= 2:
        reg_means = {}
        for reg in valid_regs:
            reg_means[reg] = np.mean(all_reg_vecs[reg], axis=0)
        
        corr_mat = np.zeros((n_reg, n_reg))
        for i, r1 in enumerate(valid_regs):
            for j, r2 in enumerate(valid_regs):
                if i == j:
                    corr_mat[i, j] = 1.0
                else:
                    # Cross-animal correlation between these two regions
                    # For each animal, get the residual from group mean
                    r1_vals = np.array(all_reg_vecs[r1])
                    r2_vals = np.array(all_reg_vecs[r2])
                    min_n = min(len(r1_vals), len(r2_vals))
                    if min_n >= 3:
                        # Use mean strength as representative metric
                        corr_mat[i, j] = np.corrcoef(r1_vals[:min_n, 1], r2_vals[:min_n, 1])[0, 1]
        
        im = ax.imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1)
        ax.set_xticks(range(n_reg))
        ax.set_xticklabels(valid_regs, fontsize=9)
        ax.set_yticks(range(n_reg))
        ax.set_yticklabels(valid_regs, fontsize=9)
        
        for i in range(n_reg):
            for j in range(n_reg):
                ax.text(j, i, f'{corr_mat[i,j]:.2f}', ha='center', va='center', fontsize=8)
        
        plt.colorbar(im, ax=ax, fraction=0.046)
    
    ax.set_title(f'{grp} (n={len(grp_animals)})', fontweight='bold',
                 color=group_colors[grp], fontsize=11)

plt.suptitle('Cross-Region Strength Correlations by Group\n'
             '(Do animals with strong AC also have strong CA1?)',
             fontsize=12, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# ANALYSIS 5: AGE x GENOTYPE INTERACTION + FULL FEATURE PCA
#
# Does the AD effect depend on age? Young AD (pre-plaque) vs Old AD (plaques)
# may show fundamentally different patterns. The interaction test asks:
# is the AD-WT difference BIGGER in old than young animals?
# =============================================================================
print('=' * 70)
print('ANALYSIS 5: Age x Genotype Interaction')
print('=' * 70)

# Test interaction on key summary metrics from each analysis
interaction_feats = []

# From Analysis 1: profile similarity (use animal-level mean)
for name, a in animals.items():
    # Overall filter consistency
    if name in filter_consistency and 'overall' in filter_consistency[name]:
        a['overall_lat_cv'] = filter_consistency[name]['overall']['lat_cv']
        a['overall_strength_cv'] = filter_consistency[name]['overall']['strength_cv']
    
    # Cross-region coordination
    if name in cross_reg_results:
        a['cross_reg_coord'] = cross_reg_results[name]['mean_corr']
    
    # Mean metrics across regions
    densities = [rd['density'] for rd in a['regions'].values()]
    strengths = [rd['mean_strength'] for rd in a['regions'].values()]
    a['mean_density'] = np.mean(densities) if densities else np.nan
    a['mean_strength'] = np.mean(strengths) if strengths else np.nan

interaction_feats = [
    ('mean_density', 'Mean connection density'),
    ('mean_strength', 'Mean coupling strength'),
    ('overall_lat_cv', 'Filter peak latency CV'),
    ('overall_strength_cv', 'Filter strength CV'),
    ('cross_reg_coord', 'Cross-region coordination'),
]

print(f'\nTest: Interaction = (Old_AD - Old_WT) - (Young_AD - Young_WT)')
print(f'Significant interaction → AD effect changes with age')

fig, axes = plt.subplots(1, len(interaction_feats), figsize=(5 * len(interaction_feats), 5))

interaction_results = []
print(f'\n{"Feature":<30} {"Yng eff":>10} {"Old eff":>10} {"Interact":>10} {"p":>8}')
print('-' * 75)

for idx, (feat, label) in enumerate(interaction_feats):
    ax = axes[idx]
    
    vals = {g: [] for g in group_order}
    for name, a in animals.items():
        v = a.get(feat, np.nan)
        if v is not None and not (isinstance(v, float) and np.isnan(v)):
            vals[a['group']].append(v)
    
    yw, ya = np.array(vals['YOUNG_WT']), np.array(vals['YOUNG_AD'])
    ow, oa = np.array(vals['OLD_WT']), np.array(vals['OLD_AD'])
    
    if any(len(v) < 2 for v in [yw, ya, ow, oa]):
        print(f'{label:<30} insufficient data')
        continue
    
    young_eff = np.mean(ya) - np.mean(yw)
    old_eff = np.mean(oa) - np.mean(ow)
    interaction = old_eff - young_eff
    
    # Permutation test for interaction
    all_vals = np.concatenate([yw, ya, ow, oa])
    sizes = [len(yw), len(ya), len(ow), len(oa)]
    n_perm = 50000
    null_inter = np.zeros(n_perm)
    
    for pi in range(n_perm):
        perm = np.random.permutation(all_vals)
        idx_p = 0
        gp = []
        for s in sizes:
            gp.append(perm[idx_p:idx_p + s])
            idx_p += s
        null_inter[pi] = (np.mean(gp[3]) - np.mean(gp[2])) - (np.mean(gp[1]) - np.mean(gp[0]))
    
    p_int = (np.sum(np.abs(null_inter) >= np.abs(interaction)) + 1) / (n_perm + 1)
    sig = '*' if p_int < 0.05 else ''
    
    print(f'{label:<30} {young_eff:>+10.4f} {old_eff:>+10.4f} {interaction:>+10.4f} {p_int:>8.4f} {sig}')
    interaction_results.append({
        'feat': feat, 'label': label,
        'young_eff': young_eff, 'old_eff': old_eff,
        'interaction': interaction, 'p': p_int
    })
    
    # Interaction plot
    for geno, color, marker in [('WT', 'steelblue', 'o'), ('AD', 'indianred', 's')]:
        y_vals = vals[f'YOUNG_{geno}']
        o_vals = vals[f'OLD_{geno}']
        means = [np.mean(y_vals), np.mean(o_vals)]
        sems = [np.std(y_vals) / np.sqrt(len(y_vals)),
                np.std(o_vals) / np.sqrt(len(o_vals))]
        
        ax.errorbar([0, 1], means, yerr=sems, marker=marker, markersize=10,
                    capsize=5, linewidth=2, label=geno, color=color)
        
        # Individual points
        jitter = np.random.normal(0, 0.03, len(y_vals))
        ax.scatter(jitter, y_vals, c=color, s=20, alpha=0.4, zorder=2)
        jitter = np.random.normal(0, 0.03, len(o_vals))
        ax.scatter(1 + jitter, o_vals, c=color, s=20, alpha=0.4, zorder=2)
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['YOUNG', 'OLD'])
    ax.set_title(f'{label}\nint p={p_int:.3f}', fontweight='bold', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Age x Genotype Interaction (non-parallel lines = interaction)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Full feature PCA: 4 groups in rich feature space ──
print('\n' + '=' * 70)
print('PCA: 4 Groups in Full Feature Space')
print('=' * 70)

# Use the standardized feature matrix from Analysis 1
if X_std.shape[0] >= 6:
    pca = PCA()
    X_pc = pca.fit_transform(X_std)
    
    print(f'Animals: {X_std.shape[0]}, Features: {X_std.shape[1]}')
    print(f'Variance explained: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, '
          f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%, '
          f'PC3={pca.explained_variance_ratio_[2]*100:.1f}%')
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    for grp in group_order:
        mask = groups == grp
        ax.scatter(X_pc[mask, 0], X_pc[mask, 1], c=group_colors[grp], s=120,
                   edgecolor='k', linewidth=0.5, marker=group_markers[grp],
                   label=grp, zorder=3)
    
    for i, name in enumerate(animal_names):
        ax.annotate(name, (X_pc[i, 0], X_pc[i, 1]), fontsize=7, alpha=0.7,
                    xytext=(4, 4), textcoords='offset points')
    
    # Confidence ellipses
    for grp in group_order:
        mask = groups == grp
        pts = X_pc[mask, :2]
        if len(pts) >= 3:
            cov = np.cov(pts.T)
            eigvals, eigvecs = np.linalg.eigh(cov)
            angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
            width = 2 * np.sqrt(5.991 * max(eigvals[1], 0.01))
            height = 2 * np.sqrt(5.991 * max(eigvals[0], 0.01))
            ell = Ellipse(xy=pts.mean(axis=0), width=width, height=height,
                          angle=angle, facecolor=group_colors[grp], alpha=0.1,
                          edgecolor=group_colors[grp], linewidth=2)
            ax.add_patch(ell)
    
    # Loading arrows
    scale = 3.0
    for j, feat in enumerate(feat_names_used):
        ax.annotate('', xy=(pca.components_[0, j] * scale, pca.components_[1, j] * scale),
                    xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
        ax.text(pca.components_[0, j] * scale * 1.15,
                pca.components_[1, j] * scale * 1.15,
                feat.replace('_', '\n'), fontsize=6, color='gray', ha='center')
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title('PCA: 4 Groups in Rich Feature Space', fontweight='bold', fontsize=13)
    ax.legend(fontsize=10)
    ax.axhline(0, color='lightgray', linewidth=0.5)
    ax.axvline(0, color='lightgray', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    
    # PERMDISP on full feature space
    for age in ['YOUNG', 'OLD']:
        wt_mask = groups == f'{age}_WT'
        ad_mask = groups == f'{age}_AD'
        if np.sum(wt_mask) >= 3 and np.sum(ad_mask) >= 3:
            wt_pts = X_pc[wt_mask]
            ad_pts = X_pc[ad_mask]
            wt_disp = np.mean(np.linalg.norm(wt_pts - wt_pts.mean(0), axis=1))
            ad_disp = np.mean(np.linalg.norm(ad_pts - ad_pts.mean(0), axis=1))
            
            obs = ad_disp - wt_disp
            all_pts = np.vstack([wt_pts, ad_pts])
            n_wt = len(wt_pts)
            null = np.zeros(10000)
            for pi in range(10000):
                perm = np.random.permutation(len(all_pts))
                pw = all_pts[perm[:n_wt]]
                pa = all_pts[perm[n_wt:]]
                null[pi] = (np.mean(np.linalg.norm(pa - pa.mean(0), axis=1)) -
                            np.mean(np.linalg.norm(pw - pw.mean(0), axis=1)))
            p_disp = (np.sum(null >= obs) + 1) / 10001
            
            print(f'{age} PERMDISP (full space): WT={wt_disp:.3f}, AD={ad_disp:.3f}, p={p_disp:.4f}')

In [ ]:
# =============================================================================
# SUMMARY: Convergent Evidence Assessment
# =============================================================================
print('=' * 90)
print('CONVERGENT EVIDENCE SUMMARY')
print('=' * 90)

print('''
Framework: Does AD disrupt the co-regulation structure of functional connectivity?

Schulz 2006: Measured 6 ion channels in identified neurons across 20 crabs.
  Found pairwise correlations r=0.91-0.97 between channels.
  Co-regulation = channels scale together within a neuron type.

Prinz 2004: 20M model networks. Valid solutions scattered in absolute space
  but constrained in ratio space. PCA shows low-dimensional structure.

Our tests:
  1. Schulz: Within-animal correlations between coupling properties
     (using hundreds of filters per animal)
  2. Filter temporal structure: consistency of synaptic dynamics
  3. Prinz: Cell-type coupling ratios — variance in ratio space
  4. Cross-region coordination: do regions covary within each animal?
  5. Interaction: does the AD effect depend on age?
''')

print('DIRECTION SCORECARD:')
print('-' * 70)

# Schulz results (Analysis 1)
if schulz_results:
    print('\nWithin-animal coupling correlations (Analysis 1):')
    for r in schulz_results:
        # Are correlations DIFFERENT in AD? (either direction is disruption)
        sig = f'p={r["p"]:.3f}' + (' *' if r['p'] < 0.05 else '')
        print(f'  {r["age"]} {r["label"][:40]}: WT r={r["wt_mean"]:+.3f}, AD r={r["ad_mean"]:+.3f} {sig}')

# CV results (Analysis 2)
if cv_results:
    print('\nFilter consistency (Analysis 2):')
    for r in cv_results:
        direction = 'AD > WT' if r['ad_mean'] > r['wt_mean'] else 'AD < WT'
        expected = 'YES' if r['ad_mean'] > r['wt_mean'] else 'no'
        sig = f'p={r["p"]:.3f}' + (' *' if r['p'] < 0.05 else '')
        print(f'  {r["age"]} {r["metric"]}: {direction} ({expected} — higher CV = disrupted) {sig}')

# Ratio variance results (Analysis 3)
if ratio_results:
    print('\nRatio variance (Analysis 3):')
    for r in ratio_results:
        vr = r.get('var_ratio', np.nan)
        if not np.isnan(vr):
            direction = 'AD > WT' if vr > 1 else 'AD < WT'
            expected = 'YES' if vr > 1 else 'no'
            sig = f'p_var={r["p_var"]:.3f}' + (' *' if r['p_var'] < 0.05 else '')
            print(f'  {r["age"]} {r["ratio"]}: var_ratio={vr:.2f}x ({expected} — higher = lost constraint) {sig}')

# Cross-region coordination (Analysis 4)
if coord_test_results:
    print('\nCross-region coordination (Analysis 4):')
    for r in coord_test_results:
        sig = f'p={r["p"]:.3f}' + (' *' if r['p'] < 0.05 else '')
        direction = 'WT > AD' if r['cliff_d'] > 0 else 'AD > WT'
        expected = 'YES' if r['cliff_d'] > 0 else 'no'
        print(f'  {r["age"]}: {direction} ({expected} — WT should be more coordinated) {sig}')

# Interaction results (Analysis 5)
if interaction_results:
    print('\nAge x Genotype interaction (Analysis 5):')
    for r in interaction_results:
        sig = f'p={r["p"]:.3f}' + (' *' if r['p'] < 0.05 else '')
        direction = 'effect LARGER in OLD' if abs(r['old_eff']) > abs(r['young_eff']) else 'effect LARGER in YOUNG'
        print(f'  {r["label"]}: {direction} {sig}')

print('\n' + '=' * 90)
print('INTERPRETATION')
print('=' * 90)

print(f'''
POWER: n=5 vs 4 per age group. Min possible p = {1/comb(9,5):.4f}. EXPLORATORY.

KEY DIFFERENCE FROM OLD NOTEBOOK:
- Old: tested mean differences with 3 summary stats per region (76 features, all NS)
- New: tests co-regulation STRUCTURE using hundreds of coupling filters per animal
  - Within-animal correlations between coupling properties (Schulz)
  - Filter temporal consistency (novel — Marder had no temporal filter data)
  - Cell-type coupling ratio variance (Prinz)
  - Cross-region coordination (Schulz PD-neuron analog)

CONVERGENT EVIDENCE matters more than any single p-value.
If OLD_AD consistently shows disrupted co-regulation across analyses
while YOUNG_AD does not, that is biologically meaningful:
  → Pathology-dependent loss of homeostatic circuit co-regulation.
''')